<a href="https://colab.research.google.com/github/rkk7452/OVRA-PyBDSF-73MHz/blob/main/Ryan_extract_sources_with_bdsf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import necessary packages
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import ZScaleInterval, ImageNormalize
import os
import pandas as pd

In [ ]:
# Install pyBDSF (https://github.com/lofar-astron/pybdsf)
%pip install bdsf

# Install a utility that we'll use to uncompress the .fits files
# Compressed files can be imaged but we can't run BDSF on them
!apt install -y libcfitsio-bin

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libcfitsio-bin is already the newest version (4.0.0-1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [ ]:
import bdsf

In [ ]:
folder_path = "/content/drive/MyDrive/Caltech SRC '26/OVRO-LWA Data/"
ryan_folder_path = "/content/drive/MyDrive/Caltech SRC '26/"
file_name = "73MHz-Clean-Snapshot-20250507_095214-image.fits"

In [ ]:
#TO DOs:

#create a list of all unique file names in this folder at 73MHz.
#Save the part between Clean-Snapshot and -image.fits in the file name into a list
#then iterate through the list and run the pybdsf and create a dataframe with the name of the file
#then write an algorithm to compare these different dataframes


In [ ]:
# Needed if running on Google Colab only
# Gives access to data stored on Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Confirm files on Google Drive
datafiles = os.listdir(folder_path)  # Change path to whatever folder holds your data
print(f"Datafile names: {datafiles}")  # Confirm that the datafiles were found

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Datafile names: ['69MHz-Clean-Snapshot-20250507_102723-image.fits.fz', '69MHz-Clean-Snapshot-20250507_102733-image.fits.fz', '69MHz-Clean-Snapshot-20250507_102743-image.fits.fz', '69MHz-Clean-Snapshot-20250507_102804-image.fits.fz', '69MHz-Clean-Snapshot-20250507_102814-image.fits.fz', '69MHz-Clean-Snapshot-20250507_102824-image.fits.fz', '69MHz-Clean-Snapshot-20250507_102835-image.fits.fz', '69MHz-Clean-Snapshot-20250507_102845-image.fits.fz', '69MHz-Clean-Snapshot-20250507_102905-image.fits.fz', '69MHz-Clean-Snapshot-20250507_102855-image.fits.fz', '69MHz-Clean-Snapshot-20250507_102926-image.fits.fz', '69MHz-Clean-Snapshot-20250507_102915-image.fits.fz', '69MHz-Clean-Snapshot-20250507_102936-image.fits.fz', '69MHz-Clean-Snapshot-20250507_102946-image.fits.fz', '69MHz-Clean-Snapshot-20250507_102956-image.fits.fz', '69MHz-Clean-Snapshot-20250507_103017-image.

In [ ]:

# Initialize the master dataframe
combined_df = pd.DataFrame(columns=['ra', 'dec', 'total_flux', 'file_name'])
combined_csv_path = f"{ryan_folder_path}combined_sources_73MHz.csv"


# creating a list of file names at 73MHz
file_list_73 = [item for item in datafiles if ("73MHz-Clean-Snapshot" in item and item.endswith("-image.fits.fz"))]
file_list_73 = [item.removesuffix(".fz") for item in file_list_73]
clean_file_list_73 = list(map(lambda x: f"{x.split('-')[0]}-{x.split('-')[3]}", file_list_73)) #file names with just frequency, date, and time (e.g. '73MHz-20250507_095214')
print(file_list_73)

['73MHz-Clean-Snapshot-20250507_095214-image.fits', '73MHz-Clean-Snapshot-20250507_095225-image.fits', '73MHz-Clean-Snapshot-20250507_095235-image.fits', '73MHz-Clean-Snapshot-20250507_095245-image.fits', '73MHz-Clean-Snapshot-20250507_101732-image.fits', '73MHz-Clean-Snapshot-20250507_101742-image.fits', '73MHz-Clean-Snapshot-20250507_101752-image.fits', '73MHz-Clean-Snapshot-20250507_101802-image.fits', '73MHz-Clean-Snapshot-20250507_101812-image.fits', '73MHz-Clean-Snapshot-20250507_101822-image.fits', '73MHz-Clean-Snapshot-20250507_101833-image.fits', '73MHz-Clean-Snapshot-20250507_101843-image.fits', '73MHz-Clean-Snapshot-20250507_101853-image.fits', '73MHz-Clean-Snapshot-20250507_101903-image.fits', '73MHz-Clean-Snapshot-20250507_101913-image.fits', '73MHz-Clean-Snapshot-20250507_101923-image.fits', '73MHz-Clean-Snapshot-20250507_101933-image.fits', '73MHz-Clean-Snapshot-20250507_101943-image.fits', '73MHz-Clean-Snapshot-20250507_101954-image.fits', '73MHz-Clean-Snapshot-20250507

In [ ]:

import shutil


# Define paths (Ensure directories end with '/')
compressed_dir = folder_path                   # Folder 1: Source (.fz files on Drive)
csv_output_path = combined_csv_path             # Folder 3: Master CSV path on Drive

# Use Colab's local runtime storage for processing
local_temp_fits = "/content/temp_processing.fits"

# Load existing progress from Drive
if os.path.exists(csv_output_path):
    print("Found existing CSV. Loading progress...")
    combined_df = pd.read_csv(csv_output_path)
    completed_files = set(combined_df['file_name'].unique())
else:
    print("No existing CSV found. Starting fresh...")
    combined_df = pd.DataFrame()
    completed_files = set()

for item in file_list_73:

    if item in completed_files:
        continue

    print(f"Processing {item}...")
    comp_file_path = os.path.join(compressed_dir, f"{item}.fz")

    try:
        # Uncompress directly to Colab's LOCAL storage
        !funpack -F -O "{local_temp_fits}" "{comp_file_path}"

        # Double check the unpack succeeded
        if not os.path.exists(local_temp_fits):
            print(f"Skipping {item}: funpack failed to create file.")
            continue

        # Run pybdsf locally
        img = bdsf.process_image(local_temp_fits, quiet=True)

        # Extract positions and fluxes
        sources = getattr(img, "sources", [])
        if not sources:
            print(f"No sources found in {item}.")
            # Even if empty, record it so we don't scan it again
            filtered_df = pd.DataFrame(columns=['ra', 'dec', 'total_flux', 'file_name'])
        else:
            ras = np.full(len(sources), np.nan)
            decs = np.full(len(sources), np.nan)
            fluxes = np.full(len(sources), np.nan)

            for source_ind, source in enumerate(sources):
                ras[source_ind] = source.posn_sky_centroid[0]
                decs[source_ind] = source.posn_sky_centroid[1]
                fluxes[source_ind] = source.total_flux

            # Clean out NaNs/Inf values
            keep_sources_inds = np.where(np.isfinite(ras) & np.isfinite(decs) & np.isfinite(fluxes))
            ras = ras[keep_sources_inds]
            decs = decs[keep_sources_inds]
            fluxes = fluxes[keep_sources_inds]

            filtered_df = pd.DataFrame({
                'ra': ras,
                'dec': decs,
                'total_flux': fluxes,
                'file_name': item
            })

        # Append data and save to Google Drive CSV
        combined_df = pd.concat([combined_df, filtered_df], ignore_index=True)
        combined_df.to_csv(csv_output_path, index=False)

        completed_files.add(item)
        print(f"Successfully processed {item}. Total sources in CSV: {len(combined_df)}")
        print(f"File number: {len(completed_files)}")

    except Exception as e:
        print(f"Error processing {item}: {e}")

    finally:
        # Clean up local space and clear RAM cache
        if 'img' in locals():
            del img # Free up pybdsf memory pointers
        if os.path.exists(local_temp_fits):
            os.remove(local_temp_fits)


print("All files processed!")

Found existing CSV. Loading progress...
Processing 73MHz-Clean-Snapshot-20250507_101732-image.fits...


INFO:PyBDSF.Process:Processing /content/temp_processing.fits
INFO:PyBDSF.Init:PyBDSF version 1.14.1
INFO:PyBDSF.Init:Non-default input parameters:
    filename             = '/content/temp_processing.fits'
    quiet                = True
USERINFO:PyBDSF.Readfile:--> Opened '/content/temp_processing.fits'
INFO:PyBDSF.Readfile:Original data shape of /content/temp_processing.fits: (1, 1, 3122, 3122)
INFO:PyBDSF.Readfile:Final data shape (npol, nchan, x, y): (1, 1, 3122, 3122)
USERINFO:PyBDSF.Readimage:Image size .............................. : (3122, 3122) pixels
USERINFO:PyBDSF.Readimage:Number of channels ...................... : 1
USERINFO:PyBDSF.Readimage:Number of Stokes parameters ............. : 1
USERINFO:PyBDSF.InitBeam:Beam shape (major, minor, pos angle) .... : (1.75811e-01, 1.14577e-01, 54.9) degrees
INFO:PyBDSF.Readimage:Equinox of image is 2000.000000.
USERINFO:PyBDSF.Collapse:Frequency of image ...................... : 75.402 MHz
USERINFO:PyBDSF.Collapse:Number of blank pi

Successfully processed 73MHz-Clean-Snapshot-20250507_101732-image.fits. Total sources in CSV: 84399
File number: 153
Processing 73MHz-Clean-Snapshot-20250507_101742-image.fits...


INFO:PyBDSF.Process:Processing /content/temp_processing.fits
INFO:PyBDSF.Init:PyBDSF version 1.14.1
INFO:PyBDSF.Init:Non-default input parameters:
    filename             = '/content/temp_processing.fits'
    quiet                = True
USERINFO:PyBDSF.Readfile:--> Opened '/content/temp_processing.fits'
INFO:PyBDSF.Readfile:Original data shape of /content/temp_processing.fits: (1, 1, 3122, 3122)
INFO:PyBDSF.Readfile:Final data shape (npol, nchan, x, y): (1, 1, 3122, 3122)
USERINFO:PyBDSF.Readimage:Image size .............................. : (3122, 3122) pixels
USERINFO:PyBDSF.Readimage:Number of channels ...................... : 1
USERINFO:PyBDSF.Readimage:Number of Stokes parameters ............. : 1
USERINFO:PyBDSF.InitBeam:Beam shape (major, minor, pos angle) .... : (1.75745e-01, 1.14604e-01, 54.9) degrees
INFO:PyBDSF.Readimage:Equinox of image is 2000.000000.
USERINFO:PyBDSF.Collapse:Frequency of image ...................... : 75.402 MHz
USERINFO:PyBDSF.Collapse:Number of blank pi

Successfully processed 73MHz-Clean-Snapshot-20250507_101742-image.fits. Total sources in CSV: 84919
File number: 154
Processing 73MHz-Clean-Snapshot-20250507_101752-image.fits...


...

Successfully processed 73MHz-Clean-Snapshot-20250507_105025-image.fits. Total sources in CSV: 173434
File number: 323
Processing 73MHz-Clean-Snapshot-20250507_105035-image.fits...


INFO:PyBDSF.Process:Processing /content/temp_processing.fits
INFO:PyBDSF.Init:PyBDSF version 1.14.1
INFO:PyBDSF.Init:Non-default input parameters:
    filename             = '/content/temp_processing.fits'
    quiet                = True
USERINFO:PyBDSF.Readfile:--> Opened '/content/temp_processing.fits'
INFO:PyBDSF.Readfile:Original data shape of /content/temp_processing.fits: (1, 1, 3122, 3122)
INFO:PyBDSF.Readfile:Final data shape (npol, nchan, x, y): (1, 1, 3122, 3122)
USERINFO:PyBDSF.Readimage:Image size .............................. : (3122, 3122) pixels
USERINFO:PyBDSF.Readimage:Number of channels ...................... : 1
USERINFO:PyBDSF.Readimage:Number of Stokes parameters ............. : 1
USERINFO:PyBDSF.InitBeam:Beam shape (major, minor, pos angle) .... : (1.79090e-01, 1.14429e-01, 50.7) degrees
INFO:PyBDSF.Readimage:Equinox of image is 2000.000000.
USERINFO:PyBDSF.Collapse:Frequency of image ...................... : 75.402 MHz
USERINFO:PyBDSF.Collapse:Number of blank pi

Successfully processed 73MHz-Clean-Snapshot-20250507_105035-image.fits. Total sources in CSV: 173940
File number: 324
Processing 73MHz-Clean-Snapshot-20250507_105045-image.fits...


INFO:PyBDSF.Process:Processing /content/temp_processing.fits
INFO:PyBDSF.Init:PyBDSF version 1.14.1
INFO:PyBDSF.Init:Non-default input parameters:
    filename             = '/content/temp_processing.fits'
    quiet                = True
USERINFO:PyBDSF.Readfile:--> Opened '/content/temp_processing.fits'
INFO:PyBDSF.Readfile:Original data shape of /content/temp_processing.fits: (1, 1, 3122, 3122)
INFO:PyBDSF.Readfile:Final data shape (npol, nchan, x, y): (1, 1, 3122, 3122)
USERINFO:PyBDSF.Readimage:Image size .............................. : (3122, 3122) pixels
USERINFO:PyBDSF.Readimage:Number of channels ...................... : 1
USERINFO:PyBDSF.Readimage:Number of Stokes parameters ............. : 1
USERINFO:PyBDSF.InitBeam:Beam shape (major, minor, pos angle) .... : (1.77740e-01, 1.13626e-01, 51.0) degrees
INFO:PyBDSF.Readimage:Equinox of image is 2000.000000.
USERINFO:PyBDSF.Collapse:Frequency of image ...................... : 75.402 MHz
USERINFO:PyBDSF.Collapse:Number of blank pi

Successfully processed 73MHz-Clean-Snapshot-20250507_105045-image.fits. Total sources in CSV: 174439
File number: 325
All files processed!



# Leftover stuff from the original demo notebook below:

In [ ]:
# Plot an image
with fits.open(folder_path+file_name+".fz") as hdul:
    data = hdul[1].data.squeeze()
    header = hdul[1].header

wcs = WCS(header).celestial

norm = ImageNormalize(data, interval=ZScaleInterval())

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection=wcs)
im = ax.imshow(data, origin="lower", cmap="Greys_r", norm=norm)

ax.set_xlabel("RA")
ax.set_ylabel("Dec")
ax.coords.grid(color="white", alpha=0.3, linestyle="solid")
fig.colorbar(im, ax=ax, label="Flux density (Jy/beam)")

plt.tight_layout()
plt.savefig("output.png", dpi=150)
plt.show()

In [ ]:
# Uncompress a data file (required to run BDSF)
# First path is the path to the uncompressed file (change to a folder in your local Drive)
# Second path is the path to the compressed file
!funpack -O "{folder_path}{file_name}" "{folder_path}{file_name}.fz"

In [ ]:
img = bdsf.process_image(
    f"{folder_path}{file_name}",
)

In [ ]:
# Plot pyBDSF output
img.show_fit()

In [ ]:
# Write the source information to a .csv file (optional)
img.write_catalog(outfile=f"{ryan_folder_path}{file_name}.csv",format="csv")

In [ ]:
# Extract the position and flux of each source
sources = getattr(img, "sources", [])
ras = np.full(len(sources), np.nan)
decs = np.full(len(sources), np.nan)
fluxes = np.full(len(sources), np.nan)
for source_ind, source in enumerate(sources):
  ras[source_ind] = source.posn_sky_centroid[0]
  decs[source_ind] = source.posn_sky_centroid[1]
  fluxes[source_ind] = source.total_flux

In [ ]:
# Note that some of the values are nan. I suspect this is because the sources
# are on or beyond the horizon (don't correspond to real astronomical sources).
print(ras[1000:1030])

In [ ]:
# Only keep sources with non-nan values
keep_sources_inds = np.where(np.isfinite(ras) & np.isfinite(decs) & np.isfinite(fluxes))
ras = ras[keep_sources_inds]
decs = decs[keep_sources_inds]
fluxes = fluxes[keep_sources_inds]

In [ ]:
# Create a DataFrame from the filtered sources
filtered_df = pd.DataFrame({
    'ra': ras,
    'dec': decs,
    'total_flux': fluxes
})

# Save the filtered data to a new CSV
output_path = f"{ryan_folder_path}{file_name}_filtered.csv"
filtered_df.to_csv(output_path, index=False)
print(f"Successfully exported {len(filtered_df)} sources to {output_path}")

In [ ]:
# Delete the uncompressed file to free space on your Drive
!rm "{folder_path}{file_name}"